In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, shutil, json

ROOT_DIR        = "/content/drive/MyDrive/nutrition_rag"
PROCESSED_DIR   = os.path.join(ROOT_DIR, "data_processed")
VECTOR_DB_DIR   = os.path.join(ROOT_DIR, "vector_db_v5")
OUTPUT_DIR      = os.path.join(ROOT_DIR, "outputs/v5")
ALL_RECORDS_FILE = os.path.join(PROCESSED_DIR, "all_records.json")

for d in [PROCESSED_DIR, VECTOR_DB_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(ALL_RECORDS_FILE):
    from google.colab import files
    print("Chọn file all_records.json để upload...")
    uploaded = files.upload()
    src = next(iter(uploaded))
    shutil.move(src, ALL_RECORDS_FILE)

print(f"✅ all_records.json: {ALL_RECORDS_FILE}")


✅ all_records.json: /content/drive/MyDrive/nutrition_rag/data_processed/all_records.json


In [ ]:
# ── Retrieval stack ───────────────────────────────────────────
!pip install -q sentence-transformers chromadb rank_bm25
!pip install -q FlagEmbedding

# ── LLM stack (BitsAndBytes thuần, không cần Unsloth) ────────
# Pin transformers < 4.56 để tránh conflict với trl/peft trên Colab
!pip install -q "transformers>=4.43.0,<4.56.0"
!pip install -q "accelerate>=0.34.0"
!pip install -q "bitsandbytes>=0.43.0"

# ── Evaluation stack ──────────────────────────────────────────
!pip install -q rouge-score nltk scikit-learn
!pip install -q google-generativeai

print("✅ Cài đặt xong.")


✅ Cài đặt xong.


In [ ]:
import gc, os, json, re, torch, numpy as np, random, time
import ast
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

# ── VRAM Guard config ─────────────────────────────────────────
VRAM_CONFIG = {
    "embed_device_indexing":  "cuda",   # embed khi build index → dùng GPU
    "embed_device_inference": "cpu",    # embed lúc query → CPU để nhường VRAM cho LLM
    "reranker_device":        "cpu",    # reranker luôn CPU
    "rerank_batch_size":      8,        # forward pass từng batch 8 cặp
    "rrf_pool":               30,       # Top-N sau RRF (giảm từ 50 → 30)
    "max_rerank_candidates":  30,
    "embed_batch_size":       64,
    "max_context_chars":      2400,
    "max_doc_chars":          800,
    "collection_name":        "nutrition_v5",
}

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_vram(label=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserv = torch.cuda.memory_reserved() / 1e9
        print(f"[VRAM {label}] Alloc: {alloc:.2f}GB | Reserved: {reserv:.2f}GB")
    else:
        print(f"[VRAM {label}] No GPU detected")

print_vram("startup")


[VRAM startup] Alloc: 6.83GB | Reserved: 7.25GB


In [ ]:

# Mapping được xây dựng dựa trên toàn bộ food_name_en trong all_records.json (526 records)
# Sắp xếp từ dài → ngắn để tránh partial-match conflict (vd: "thịt bò" trước "bò")
VI_EN_FOOD_MAP = {
    # === NGŨ CỐC / TINH BỘT ===
    "gạo nếp":         "glutinous rice",
    "nếp xay":         "glutinous rice, milled",
    "gạo lứt":         "rice, brown or hulled",
    "gạo trắng":       "ordinary polished rice",
    "cơm trắng":       "ordinary polished rice",
    "gạo chưa xay":    "under milled, home-pounded rice",
    "gạo non":         "rice, unriped",
    "bột mì":          "wheat flour",
    "bột gạo nếp":     "glutinous rice flour",
    "bột gạo":         "rice ordinary flour",
    "bột ngô":         "yellow maize flour",
    "bột sắn":         "cassava flour",
    "bột khoai lang":  "sweet potato flour",
    "bột dong":        "canna edulis ker, flour",
    "bánh mì":         "french bread",
    "bánh phở":        "rice noodles",
    "bánh gạo":        "rice cake, plain",
    "bánh tráng":      "rice paper for rollers",
    "bánh cuốn":       "rice paper for rollers",
    "bún":             "rice vermicelli",
    "mì tôm":          "wheat noodles",
    "mì":              "wheat noodles",
    "miến dong":       "vermicelli from bermuda",
    "miến":            "vermicelli from bermuda",
    "ngô bắp":         "fresh maize seeds",
    "ngô khô":         "yellow maize, dried seeds",
    "ngô":             "maize",
    "bắp":             "maize",
    "khoai lang vàng": "sweet potato, yellow",
    "khoai lang trắng":"sweet potato, white",
    "khoai lang":      "sweet potato",
    "khoai tây":       "potato, white",
    "khoai mì":        "cassava",
    "sắn đắng":        "bitter cassava",
    "sắn":             "cassava",
    "khoai sọ":        "taro tuber",
    "khoai nước":      "water-taro",
    "khoai từ":        "chinese yam",
    "củ từ":           "yam winged",
    "dong riềng":      "canna edulis",
    "củ năng":         "water-caltrops",
    "sen":             "lotus",
    "hạt sen":         "lotus seed",
    "ngó sen":         "lotus, stem",
    "măng":            "bamboo shoot",
    # === ĐẬU / HẠT ===
    "đậu đen":         "black bean seeds",
    "đậu xanh":        "mungo bean seeds",
    "giá đỗ xanh":     "mungobean sprouts",
    "giá đỗ":          "mungobean sprouts",
    "giá":             "mungobean sprouts",
    "đậu phộng rang":  "cashew, common, roasted",
    "đậu phộng":       "peanut",
    "lạc rang":        "peanut, oil fried",
    "lạc":             "peanut",
    "bột lạc":         "peanut flour",
    "đậu tương":       "soybean",
    "đậu nành":        "soybean",
    "đậu hũ chiên":    "curd tofu, fried",
    "đậu hũ":          "curd tofu",
    "đậu phụ chiên":   "curd tofu, fried",
    "đậu phụ":         "curd tofu",
    "sữa đậu nành":    "soybean milk",
    "đậu đỏ":          "kidney bean",
    "đậu hà lan":      "peas garden",
    "đậu đũa":         "cow-peas",
    "đậu cove":        "french bean",
    "hạt dẻ":          "chestnut, chinese",
    "hạt điều":        "cashew nut",
    "hạt mè":          "sesame oriental seeds",
    "vừng":            "sesame oriental seeds",
    "hạt bí":          "pumpkin seeds",
    "hạt dưa hấu":     "water melon seeds",
    # === NẤM / TẢNG BIỂN ===
    "nấm hương khô":   "mushroom chinese, dried",
    "nấm hương":       "mushroom, chinese, raw",
    "nấm rơm":         "mushroom, straw",
    "nấm mèo khô":     "jew's ear, dried",
    "nấm mèo":         "jew's ear",
    "nấm thường":      "mushroom, common",
    "rong biển khô":   "dried seaweed",
    "rong biển":       "seaweed fresh",
    # === RAU CỦ ===
    "bí đỏ lá":        "pumpkin leaves",
    "bí đỏ":           "pumpkin squash",
    "bí xanh":         "asgourd waxgoured",
    "bí đao":          "asgourd waxgoured",
    "bầu":             "calabash, bottle gourd",
    "mướp đắng":       "balsam-pear",
    "khổ qua":         "balsam-pear",
    "mướp":            "gourd, sponge gourd",
    "su su":           "chayote",
    "cà tím nhỏ":      "egg plant - small",
    "cà tím":          "egg plant big",
    "cà chua":         "tomato",
    "cà rốt khô":      "dried carrot",
    "cà rốt":          "carrots",
    "bắp cải đỏ":      "cabbage, red",
    "bắp cải":         "cabbage, common",
    "cải thảo":        "chinese cabbage, white",
    "cải xoăn":        "cauliflower, green",
    "súp lơ trắng":    "cauliflower, white",
    "súp lơ xanh":     "cauliflower, green",
    "súp lơ":          "cauliflower",
    "cải xanh dưa":    "mustard green, pickled",
    "cải xanh":        "mustard greens",
    "rau muống khô":   "water spinach, dried",
    "rau muống":       "swamp cabbage",
    "rau ngót khô":    "sauropus, dried",
    "rau ngót":        "sauropus",
    "rau má":          "wort, india penny",
    "rau đay":         "jute potherb",
    "rau dền trắng":   "amaranth, sp white",
    "rau dền đỏ":      "amaranth, sp. red",
    "rau dền":         "amaranth",
    "rau cần":         "celery, chinese",
    "xà lách":         "lettuce garden",
    "rau mùi":         "coriander",
    "rau thơm":        "coriander",
    "húng quế":        "basil sweet leaves",
    "húng":            "mint leaves",
    "tía tô":          "perilla",
    "hành tây dưa":    "onion, pickled",
    "hành tây":        "onion, common, garden",
    "hành lá":         "onion, welsh",
    "hành phi":        "onion, fragrant",
    "tỏi":             "garlic bulbs",
    "gừng bột":        "ginger root, dried powder",
    "gừng":            "ginger root, fresh",
    "nghệ khô":        "turmeric rhizome, dried powder",
    "nghệ":            "turmeric, rhizome, fresh",
    "ớt đỏ":           "chili pepper",
    "ớt bột":          "red pepper, dried powder",
    "ớt xanh":         "peppers, green",
    "ớt vàng":         "peppers, yellow",
    "dưa chuột dưa":   "cucumber, pickled",
    "dưa leo":         "cucumber",
    "dưa chuột":       "cucumber",
    "cà chua dưa":     "tomato, pickled",
    "củ cải trắng":    "radish garden white",
    "củ cải đỏ":       "red radish oriental",
    "củ cải khô":      "dried radish, white",
    "su hào":          "kohlrabi",
    "su hào khô":      "dried kohlrabi",
    "măng tây trắng":  "asparagus, white",
    "bí bắp cải":      "cabbage chinese, pickled",
    "rau ngổ":         "limnophila aromatic",
    "diếp cá":         "polygonum odoratum",
    "lá lốt":          "lolot",
    # === TRÁI CÂY ===
    "chuối chín":      "banana",
    "chuối xanh":      "banana common varieties, unripe",
    "chuối":           "banana",
    "cam":             "orange",
    "nước cam":        "orange juice",
    "quýt":            "tangerine",
    "mandarin":        "mandarin",
    "bưởi":            "pomelo, pummelo",
    "chanh":           "lemon",
    "dứa dại":         "pineapple, wild",
    "dứa":             "pineapple",
    "thơm":            "pineapple",
    "xoài chín":       "mango, common; india mango, ripe",
    "xoài xanh":       "mango, common; indian mango, unripe",
    "xoài":            "mango",
    "ổi":              "guava, common",
    "đu đủ chín":      "papaya, ripe",
    "đu đủ xanh":      "papaya, unripe",
    "đu đủ":           "papaya",
    "dưa hấu":         "watermelon",
    "dưa lưới":        "musk melon",
    "dưa vàng":        "honey dew melon",
    "nho":             "grape",
    "nhãn":            "longan",
    "nhãn khô":        "longan, dried",
    "vải":             "litchi",
    "vải khô":         "litchi, dried",
    "chôm chôm":       "rambutan",
    "mận":             "japanese, plum",
    "đào":             "peach",
    "lê":              "pear",
    "táo":             "apple, common, domestic",
    "mít":             "jackfruit",
    "sầu riêng":       "durian",
    "bơ tươi":         "avocado, green",
    "bơ tím":          "avocado, purple",
    "bơ":              "avocado",
    "thanh long":      "dragon's eyes fruit",
    "khế":             "carambola",
    "na":              "sugarapple, sweetsop",
    "hồng":            "persimmon kaki",
    "me":              "tamarind fruit",
    "gấc":             "gac fruit",
    "kiwi":            "kiwi fruit",
    "dâu tây":         "strawberry",
    "dâu đen":         "blackberry",
    "mơ khô":          "apricot dried",
    "táo tàu":         "jujube, common or chinese",
    # === THỊT ===
    "thịt bò loại i":   "beef, grade i",
    "thịt bò loại ii":  "beef, grade ii",
    "thịt bò loại 1":   "beef, grade i",
    "thịt bò loại 2":   "beef, grade ii",
    "thịt bò sấy":      "dried beef",
    "thịt bò hộp":      "beef, canned",
    "thịt bò":          "beef",
    "bò":               "beef",
    "thịt lợn nạc mỡ": "pork, lean and fat",
    "thịt lợn nạc":     "pork, lean",
    "thịt lợn mỡ":      "pork, medium fat",
    "thịt lợn sườn":    "pork, ribs",
    "thịt lợn đùi":     "pork, leg",
    "thịt lợn hộp":     "pork, canned",
    "thịt lợn":         "pork",
    "thịt heo nạc":     "pork, lean",
    "thịt heo":         "pork",
    "heo":              "pork",
    "lợn":              "pork",
    "giăm bông":        "ham, pork",
    "xúc xích lợn":     "pork sausage",
    "xúc xích trung quốc": "chinese sausage",
    "da lợn":           "pork skin",
    "thịt gà ta":       "chicken meat, average",
    "thịt gà hộp":      "chicken, canned",
    "thịt gà":          "chicken",
    "gà":               "chicken",
    "thịt vịt":         "duck meat, average",
    "vịt":              "duck meat",
    "thịt ngỗng":       "goose",
    "thịt dê":          "goat, meat, lean",
    "dê":               "goat",
    "thịt cừu":         "mutton meat, lean",
    "thịt trâu nạc":    "buffalo meat, lean",
    "thịt trâu khô":    "buffalo meat, dried",
    "thịt trâu":        "buffalo meat, average",
    "trâu":             "buffalo meat",
    "thịt bê nạc mỡ":   "veal meat, lean and fat",
    "thịt bê":          "veal meat",
    "bê":               "veal",
    "thịt ngựa":        "horse meat",
    "thịt thỏ":         "rabbit meat",
    "thịt chó đùi":     "dog, shoulder",
    "thịt chó":         "dog meat",
    "thịt nai":         "deer meat",
    "thịt gà tây":      "turkey raw flesh",
    "thịt chim bồ câu": "pigeon young bird",
    # === PHỦ TẠNG ===
    "gan bò":           "beef, liver",
    "tim bò":           "beef, heart",
    "thận bò":          "beef, kidney",
    "phổi bò":          "beef, lung",
    "lưỡi bò":          "beef tongue",
    "dạ dày bò":        "stomach, beef",
    "óc bò":            "brain beef",
    "tiết bò":          "beef, blood",
    "đuôi bò":          "beef, tail",
    "tủy bò":           "beef bone marrow",
    "đầu bò":           "head beef",
    "gan lợn":          "pork liver",
    "gan heo":          "pork liver",
    "tim lợn":          "hog heart",
    "thận lợn":         "pork, kidney",
    "phổi lợn":         "hog lung",
    "lòng lợn":         "hog intestine",
    "dạ dày lợn":       "stomach, hog",
    "óc lợn":           "hog brain",
    "tiết lợn":         "hog blood",
    "tai lợn":          "hog ears",
    "đầu lợn":          "hog, head",
    "lưỡi lợn":         "hog, tongue",
    "đuôi lợn":         "hog, tail",
    "tủy lợn":          "hog, bone marrow",
    "mề gà":            "chicken gizzard",
    "tim gà":           "chicken heart",
    "gan gà":           "chicken liver",
    "lòng gà":          "chicken giblets",
    "gan vịt":          "duck liver",
    # === CÁ / HẢI SẢN ===
    "cá chép":          "carp",
    "cá trắm":          "carp, amur",
    "cá rô phi":        "tilapia",
    "cá trôi":          "major carp",
    "cá rô":            "anabas",
    "cá lóc":           "fish, snake head",
    "cá quả":           "fish, snake head",
    "cá hồi":           "salmon",
    "cá thu":           "mackerel",
    "cá ngừ":           "flying fish, tuna",
    "cá mòi":           "sardin",
    "cá trích":         "herring",
    "cá chình":         "eel",
    "cá da trơn":       "catfish",
    "cá bơn":           "flounder",
    "cá đối":           "mullet",
    "cá bống":          "goby",
    "cá cơm":           "scad, anchovy",
    "cá khô":           "dried fish",
    "cá mắm":           "shredded snake-head fish, salted",
    "tôm biển":         "sea shrimp",
    "tôm khô":          "shrimp, dried",
    "tôm moi":          "tiny shrimp",
    "tôm tép":          "fresh-water shrimp",
    "tôm":              "shrimp",
    "cua biển":         "crab, sea water",
    "cua đồng":         "crab, fresh water",
    "ghẹ":              "small sea - crab",
    "mực khô":          "dried cuttle fish",
    "mực":              "cuttle fish, raw",
    "bạch tuộc":        "cuttle fish",
    "nghêu":            "clam",
    "hàu":              "oyster",
    "ốc sên":           "snail large",
    "ốc":               "snail medium",
    "sứa":              "sea slug",
    "ếch":              "frog",
    "tằm":              "silk worm",
    "châu chấu":        "locust",
    # === TRỨNG ===
    "trứng gà bột":     "chicken egg powder",
    "trứng gà lòng trắng": "hen egg, white",
    "trứng gà lòng đỏ":  "hen egg, yolk",
    "trứng gà":         "hen egg, raw, whole",
    "trứng vịt lộn":    "duck egg, embryonated",
    "trứng vịt lòng trắng": "duck egg, white",
    "trứng vịt lòng đỏ": "duck egg, yolk",
    "trứng vịt":        "duck egg, whole",
    "trứng cút":        "quail egg",
    # === SỮA ===
    "sữa mẹ":           "breast milk",
    "sữa bột nguyên kem": "milk powder, whole",
    "sữa bột tách béo": "skimmed milk powder",
    "sữa đặc có đường": "milk condensed, sweetened",
    "sữa bột đậu":      "milk flour, made from roasted soybeans",
    "sữa bột":          "milk powder",
    "sữa chua sôcôla":  "yogurt, chocolate",
    "sữa chua":         "yogurt",
    "sữa tươi bò":      "milk cow, fresh",
    "sữa bò":           "milk cow",
    "sữa tươi":         "milk cow, fresh",
    "sữa dê":           "milk goat's",
    "phô mai":          "cheese",
    # === DẦU MỠ ===
    "bơ động vật":      "butter, unsalted",
    "bơ thực vật":      "butter-margarine blend",
    "mỡ lợn lỏng":      "lard, liquid",
    "mỡ lợn":           "lard",
    "dầu dừa":          "coconut oil",
    "dầu mè":           "sesame oil",
    "dầu đậu phộng":    "peanut oil",
    "dầu đậu nành":     "soybean oil",
    "dầu ngô":          "corn oil",
    "dầu ô liu":        "olive oil",
    "dầu cọ":           "palm oil",
    "dầu cám":          "rice bran oil",
    "dầu hạt bông":     "cottonseed oil",
    "dầu thực vật":     "vegetable oil",
    "sốt mayonnaise":   "mayonnaise",
    # === GIA VỊ ===
    "đường trắng":      "refined sugar",
    "mật ong":          "honey",
    "muối":             "table salt",
    "tiêu":             "peppercorn",
    "ớt khô bột":       "red pepper, dried powder",
    "nước mắm thượng hạng": "fish - sauce (super quality)",
    "nước mắm hạng 1":  "fish sauce, grade i",
    "nước mắm hạng 2":  "fish sauce, grade ii",
    "nước mắm cô":      "fish sauce, concentrated",
    "nước mắm":         "fish sauce",
    "tương đậu":        "soybean sauce",
    "tương nếp":        "soybean sauce with glutinous rice",
    "mắm tôm chiên":    "deep fried shrimp paste",
    "mắm tôm":          "shrimp paste",
    "mắm ruốc":         "shrimp sauce",
    "cà ri":            "cari powder",
    # === ĐỒ UỐNG ===
    "cà phê":           "coffee",
    "ca cao":           "cocoa powder",
    "bia":              "beer light",
    "rượu vang đỏ":     "red wine",
    "rượu vang trắng":  "white wine",
    "rượu vang":        "red wine",
    "nước dừa":         "coconut milk",
    "nước cam tươi":    "orange juice, fresh",
    "nước cà chua":     "tomato juice",
    "coca cola":        "coca cola",
    "vodka":            "vodka",
    "whisky":           "whisky liqueur",
}

# Sort by length desc để tránh partial-match (khớp "thịt bò loại i" trước "thịt bò")
_SORTED_MAP = sorted(VI_EN_FOOD_MAP.items(), key=lambda x: len(x[0]), reverse=True)

def translate_vi_food_terms(text: str) -> str:
    result = text.lower()
    for vi_term, en_term in _SORTED_MAP:
        pattern = r"\b" + re.escape(vi_term) + r"\b"
        if re.search(pattern, result, re.IGNORECASE):
            result = re.sub(pattern, en_term, result, flags=re.IGNORECASE)
    return result

def detect_language(text: str) -> str:
    vn_chars = r'[àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]'
    return "vi" if re.search(vn_chars, text, re.IGNORECASE) else "en"

# ── Smoke test ────────────────────────────────────────────────
tests = [
    "thịt bò loại i chứa bao nhiêu protein",
    "gan lợn nhiều sắt không",
    "trứng gà bao nhiêu calo",
    "cá hồi và cá thu loại nào nhiều omega không",
]
print("=== Translation smoke test ===")
for t in tests:
    print(f"  VI: {t}")
    print(f"  EN: {translate_vi_food_terms(t)}")
    print()


=== Translation smoke test ===
  VI: thịt bò loại i chứa bao nhiêu protein
  EN: beef, grade i chứa bao nhiêu protein

  VI: gan lợn nhiều sắt không
  EN: pork liver nhiều sắt không

  VI: trứng gà bao nhiêu calo
  EN: hen egg, raw, whole bao nhiêu calo

  VI: cá hồi và cá thu loại nào nhiều omega không
  EN: salmon và mackerel loại nào nhiều omega không



In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ── Load records ──────────────────────────────────────────────
with open(ALL_RECORDS_FILE, "r", encoding="utf-8") as f:
    all_records = json.load(f)
print(f"Loaded {len(all_records)} records")

ids       = [r["record_id"]          for r in all_records]
documents = [r["text_for_embedding"] for r in all_records]

# ── Enrich documents: thêm tên Việt để BM25 khớp được VI queries ──
def enrich_text(record: dict) -> str:
    """
    Gắn tên Việt vào đầu text_for_embedding.
    BM25 tokenize sẽ tìm thấy 'thịt bò' trong corpus thay vì chỉ 'beef'.
    """
    base = record["text_for_embedding"]
    en = str(record.get("food_name_en", "")).lower()
    # Tìm tên Việt tương ứng qua reverse lookup
    vi_names = [vi for vi, en_term in VI_EN_FOOD_MAP.items()
                if en_term.lower() in en and len(vi) > 3]
    # Giữ tên Việt ngắn nhất (generic nhất)
    vi_names = sorted(vi_names, key=len)[:3]
    if vi_names:
        vi_label = " / ".join(vi_names)
        return f"Tên tiếng Việt: {vi_label}\n{base}"
    return base

enriched_documents = [enrich_text(r) for r in all_records]
print(f"Sample enriched doc:\n{enriched_documents[0][:200]}")

# ── Embedding model: BAAI/bge-m3 ─────────────────────────────
print("\nĐang tải BAAI/bge-m3 (sẽ dùng GPU khi build index, CPU khi query)...")
embed_model = SentenceTransformer(
    "BAAI/bge-m3",
    device=VRAM_CONFIG["embed_device_indexing"]
)
print_vram("after bge-m3 load")

# ── ChromaDB ──────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = chroma_client.get_or_create_collection(
    name=VRAM_CONFIG["collection_name"],
    metadata={"hnsw:space": "cosine"}
)

if collection.count() > 0:
    print(f"ChromaDB: {collection.count()} records đã có → skip re-embed")
else:
    print("ChromaDB trống → bắt đầu embedding...")
    metadatas = []
    for r in all_records:
        m = {"record_type": r.get("record_type",""), "source": r.get("source","")}
        for k in ("group","age","food_name_en"):
            if r.get(k): m[k] = str(r[k])
        metadatas.append(m)

    BS = VRAM_CONFIG["embed_batch_size"]
    for i in tqdm(range(0, len(ids), BS)):
        s, e = i, i+BS
        embs = embed_model.encode(enriched_documents[s:e], convert_to_tensor=False).tolist()
        collection.upsert(ids=ids[s:e], documents=enriched_documents[s:e],
                          embeddings=embs, metadatas=metadatas[s:e])
    print(f"✅ {collection.count()} vectors → ChromaDB")

# Sau khi build index: chuyển embed_model sang CPU để nhường VRAM cho LLM
embed_model = SentenceTransformer("BAAI/bge-m3", device=VRAM_CONFIG["embed_device_inference"])
clear_vram()
print_vram("after moving embed to CPU")

# ── BM25 trên enriched corpus ─────────────────────────────────
print("Building BM25...")
tokenized_corpus = [doc.lower().split() for doc in enriched_documents]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 ready")


Loaded 583 records
Sample enriched doc:
Tên tiếng Việt: gạo nếp / nếp xay
Food name: Glutinous rice, milled
Calories: 344.0 kcal
Protein: 8.6 g
Fat: 1.5 g
Carbohydrate: 74.5 g
Fiber: 0.6 g
Calcium: 32.0 mg
Iron: 1.2 mg

Đang tải BAAI/bge-m3 (sẽ dùng GPU khi build index, CPU khi query)...
[VRAM after bge-m3 load] Alloc: 9.10GB | Reserved: 9.14GB
ChromaDB: 583 records đã có → skip re-embed
[VRAM after moving embed to CPU] Alloc: 6.83GB | Reserved: 7.23GB
Building BM25...
✅ BM25 ready


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch

MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print(f"Đang tải {MODEL_ID} (4-bit NF4 BitsAndBytes)...")
print_vram("before LLM load")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print("✅ Llama-3 sẵn sàng.")
print_vram("after LLM load")


def llm_call(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.1,
) -> str:
    """Gọi Llama-3 với temperature thấp, trả về text đã strip."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# Smoke test
resp = llm_call(
    "You are helpful.",
    "Say hello in one sentence.",
    max_new_tokens=30,
)

print(f"Smoke test: {resp}")

Đang tải NousResearch/Meta-Llama-3-8B-Instruct (4-bit NF4 BitsAndBytes)...
[VRAM before LLM load] Alloc: 6.83GB | Reserved: 7.23GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Llama-3 sẵn sàng.
[VRAM after LLM load] Alloc: 10.43GB | Reserved: 13.81GB
Smoke test: Hello!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
ROUTE_CONFIG = {
    "LOOKUP":    {"top_k": 3},
    "COMPARE":   {"top_k": 8},
    "RECOMMEND": {"top_k": 10},
    "EXPLAIN":   {"top_k": 5},
}

def route_query(query: str) -> str:
    lang = detect_language(query)
    if lang == "vi":
        system = (
            "Phân loại câu hỏi vào 1 trong 4 loại:\n"
            "- LOOKUP: tra cứu số liệu 1 thực phẩm.\n"
            "- COMPARE: so sánh 2+ thực phẩm.\n"
            "- RECOMMEND: gợi ý thực đơn.\n"
            "- EXPLAIN: giải thích khái niệm.\n"
            "Chỉ trả về 1 từ duy nhất."
        )
    else:
        system = (
            "Classify the question into one of: LOOKUP, COMPARE, RECOMMEND, EXPLAIN.\n"
            "Return only the one-word label."
        )
    raw = llm_call(system, query, max_new_tokens=8, temperature=0.0)
    for intent in ["LOOKUP","COMPARE","RECOMMEND","EXPLAIN"]:
        if intent in raw.upper():
            return intent
    return "LOOKUP"

print(route_query("Thịt bò grade I có bao nhiêu protein?"))
print(route_query("So sánh cá hồi và cá thu"))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LOOKUP
COMPARE


In [ ]:
import numpy as np

def decompose_query(query: str) -> list:
    lang = detect_language(query)

    if lang == "vi":
        system = (
            "Nhiệm vụ: Tách câu hỏi so sánh thành các câu hỏi tra cứu độc lập.\n"
            "KHÔNG trả lời câu hỏi. CHỈ tách thành các câu hỏi nhỏ hơn.\n"
            "Mỗi câu hỏi con trên 1 dòng, bắt đầu bằng dấu '-'.\n"
            "Nếu câu hỏi đơn giản (chỉ 1 thực phẩm), trả về đúng câu gốc.\n\n"
            "Ví dụ:\n"
            "Input: Thịt bò và thịt heo loại nào nhiều đạm hơn?\n"
            "Output:\n"
            "- Thịt bò có bao nhiêu gam đạm (protein) trên 100g?\n"
            "- Thịt heo có bao nhiêu gam đạm (protein) trên 100g?\n\n"
            "Input: Cơm trắng 100g có bao nhiêu calo?\n"
            "Output:\n"
            "- Cơm trắng 100g có bao nhiêu calo?\n"
        )
    else:
        system = (
            "Task: Split comparison questions into independent lookup sub-questions.\n"
            "Do NOT answer the question. ONLY split into smaller questions.\n"
            "Each sub-question on its own line starting with '-'.\n"
            "If the question is simple (only 1 food), return the original question.\n\n"
            "Example:\n"
            "Input: Which has more protein, beef or pork?\n"
            "Output:\n"
            "- How many grams of protein per 100g does beef have?\n"
            "- How many grams of protein per 100g does pork have?\n"
        )

    raw = llm_call(system, query, max_new_tokens=120, temperature=0.0)

    subs = [
        l.strip("- •").strip()
        for l in raw.splitlines()
        if len(l.strip()) > 5
        and not any(keyword in l.lower() for keyword in ["output:", "input:", "ví dụ", "example"])
    ][:5]

    return subs if subs else [query]


def generate_hyde_doc(query_en: str) -> str:
    """Luôn sinh bằng tiếng Anh để match English corpus."""

    system = (
        "You are a nutrition expert. Write ~50 words in English describing "
        "the typical nutritional values (calories, protein, fat, carbs) of the food asked about. "
        "Only write the description."
    )

    return llm_call(system, query_en, max_new_tokens=90, temperature=0.1)


def get_query_embedding(text: str) -> list:
    """Encode trên CPU (embed_model đã chuyển sang CPU sau khi build index)."""
    return embed_model.encode(text, convert_to_tensor=False).tolist()


def get_hyde_embedding(query_en: str, use_hyde: bool = True):
    q_emb = np.array(get_query_embedding(query_en))

    if use_hyde:
        hyde = generate_hyde_doc(query_en)
        h_emb = np.array(get_query_embedding(hyde))
        return ((q_emb + h_emb) / 2.0).tolist(), hyde

    return q_emb.tolist(), None


# Test
subs = decompose_query("So sánh lượng đạm trong thịt bò loại I và ức gà, và ghi rõ từng nguồn dinh dưỡng của tụi nó ra")
print("Decompose:", subs)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Decompose: ['Thịt bò loại I có bao nhiêu gam đạm (protein) trên 100g?', 'Uc gà có bao nhiêu gam đạm (protein) trên 100g?', 'Nguồn dinh dưỡng của thịt bò loại I là gì?', 'Nguồn dinh dưỡng của ức gà là gì?']


In [ ]:
from FlagEmbedding import FlagReranker

print("Đang tải bge-reranker-base (CPU)...")
reranker = FlagReranker("BAAI/bge-reranker-base",
                         use_fp16=False,
                         device=VRAM_CONFIG["reranker_device"])
print("✅ Reranker ready")
print_vram("after reranker")

def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
    scores = {}
    for rank, did in enumerate(dense_ids):
        scores[did] = scores.get(did, 0) + 1/(k+rank+1)
    for rank, did in enumerate(sparse_ids):
        scores[did] = scores.get(did, 0) + 1/(k+rank+1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def rerank_documents(query: str, candidate_ids: list, top_n: int = 5) -> list:
    if not candidate_ids:
        return []
    cands = candidate_ids[:VRAM_CONFIG["max_rerank_candidates"]]
    res   = collection.get(ids=cands)
    id2txt = {res["ids"][i]: res["documents"][i] for i in range(len(res["ids"]))}

    all_scores = []
    bs = VRAM_CONFIG["rerank_batch_size"]
    valid_ids = [d for d in cands if d in id2txt]
    for i in range(0, len(valid_ids), bs):
        batch = valid_ids[i:i+bs]
        pairs = [(query, id2txt[d][:VRAM_CONFIG["max_doc_chars"]]) for d in batch]
        all_scores.extend(zip(batch, reranker.compute_score(pairs)))
    all_scores.sort(key=lambda x: x[1], reverse=True)
    return [did for did, _ in all_scores[:top_n]]

def retrieve_top_ids(query: str, variant: str = "full", top_k: int = 10, use_hyde: bool = True, rerank: bool = True) -> list:
    lang = detect_language(query)
    en_query = translate_vi_food_terms(query) if lang == "vi" else query

    # 1. BM25 IDs
    bm25_scores = bm25.get_scores(en_query.lower().split())
    sparse_ids = [ids[i] for i in np.argsort(bm25_scores)[::-1][:VRAM_CONFIG["rrf_pool"]]]

    # 2. Dense IDs
    q_emb, _ = get_hyde_embedding(en_query, use_hyde=use_hyde)
    dense_res = collection.query(
        query_embeddings=[q_emb],
        n_results=min(VRAM_CONFIG["rrf_pool"], collection.count())
    )
    dense_ids = dense_res["ids"][0]

    # Resolve variant
    if variant == "bm25":
        cands = sparse_ids[:top_k]
    elif variant == "dense":
        cands = dense_ids[:top_k]
    elif variant == "rrf":
        rrf = reciprocal_rank_fusion(dense_ids, sparse_ids)
        cands = [d for d, _ in rrf[:top_k]]
    else: # full
        rrf = reciprocal_rank_fusion(dense_ids, sparse_ids)
        cands_rrf = [d for d, _ in rrf[:VRAM_CONFIG["rrf_pool"]]]
        cands = rerank_documents(en_query, cands_rrf, top_n=top_k) if rerank else cands_rrf[:top_k]

    return cands

def hybrid_search(query: str, top_k: int = 5, use_hyde: bool = True, rerank: bool = True) -> str:
    top_ids = retrieve_top_ids(query, variant="full", top_k=top_k, use_hyde=use_hyde, rerank=rerank)

    if not top_ids:
        return ""
    data = collection.get(ids=top_ids)
    id2doc = {data["ids"][i]: (data["documents"][i], data["metadatas"][i])
              for i in range(len(data["ids"]))}
    parts = []
    for i, did in enumerate(top_ids, 1):
        if did in id2doc:
            txt, meta = id2doc[did]
            parts.append(
                f"--- Doc {i} (ID:{did}|Src:{meta.get('source','?')}) ---\n"
                + txt[:VRAM_CONFIG["max_doc_chars"]]
            )
    return "\n\n".join(parts)

# Test
ctx = hybrid_search("Thịt bò loại I chứa bao nhiêu protein?", top_k=3)
print(ctx[:600])


Đang tải bge-reranker-base (CPU)...
✅ Reranker ready
[VRAM after reranker] Alloc: 5.71GB | Reserved: 13.58GB


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


--- Doc 1 (ID:food_0281|Src:food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò / thịt bò loại i / thịt bò loại 1
Food name: Beef, grade I
Calories: 118.0 kcal
Protein: 21.0 g
Fat: 3.8 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 12.0 mg
Iron: 3.1 mg

--- Doc 2 (ID:food_0282|Src:food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò / thịt bò loại i / thịt bò loại 1
Food name: Beef, grade II
Calories: 167.0 kcal
Protein: 18.0 g
Fat: 10.5 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 10.0 mg
Iron: 2.7 mg

--- Doc 3 (ID:food_0502|Src:food_composition_vn_2007) ---
Tên tiếng Việt: nước mắm / nước mắ


In [ ]:
STRICT_SYSTEM_VI = """Bạn là chuyên gia dinh dưỡng AI. Chỉ được dùng thông tin từ [CONTEXT].
QUY TẮC:
1. TUYỆT ĐỐI không tự bịa hoặc suy diễn số liệu ngoài [CONTEXT].
2. Mọi con số (kcal, protein...) PHẢI trích từ [CONTEXT] kèm đơn vị.
3. Nếu [CONTEXT] không có thông tin → trả lời: "Không có thông tin trong tài liệu."
4. Trả lời bằng tiếng Việt, ngắn gọn.
5. Nếu thực phẩm hỏi là dạng chín (cơm, cháo...) nhưng [CONTEXT] chỉ có dạng khô/sống, hãy ghi rõ: "(lưu ý: giá trị trên là dạng khô/sống, cơm/món chín sẽ thấp hơn do hút nước)"""

STRICT_SYSTEM_EN = """You are a nutrition AI expert. Only use information from [CONTEXT].
RULES:
1. NEVER fabricate numbers outside [CONTEXT].
2. Every number (kcal, protein...) MUST come from [CONTEXT] with units.
3. If [CONTEXT] lacks the info → reply: "No information available in the documents."
4. Reply in English, concise.
5. If the food asked is cooked (rice, porridge...) but [CONTEXT] is only available in dry/raw form, please clearly state: "(note: the above value is dry/raw form, cooked rice/dish will be lower due to water absorption)"""

def full_rag_answer(query: str, verbose: bool = True) -> str:
    if verbose: print(f"\n👤 {query}")

    lang    = detect_language(query)
    intent  = route_query(query)
    top_k   = ROUTE_CONFIG.get(intent, {"top_k":4})["top_k"]
    if verbose: print(f"⚙️  Intent:{intent} | Lang:{lang} | top_k:{top_k}")


    sub_queries = decompose_query(query)

    if verbose and len(sub_queries)>1:
        print(f"🔀 Decomposed: {sub_queries}")

    contexts = []
    for sq in sub_queries:
        use_hyde = intent != "LOOKUP"
        ctx = hybrid_search(sq, top_k=top_k, use_hyde=use_hyde, rerank=True)
        if ctx: contexts.append(ctx)

    combined = "\n\n".join(contexts)
    if not combined:
        return "Không tìm thấy thông tin phù hợp." if lang=="vi" else "No relevant information found."

    combined = combined[:VRAM_CONFIG["max_context_chars"]]
    system   = STRICT_SYSTEM_VI if lang=="vi" else STRICT_SYSTEM_EN
    prompt   = f"[CONTEXT]\n{combined}\n\n[CÂU HỎI]\n{query}\n\n[CÂU TRẢ LỜI]"
    answer   = llm_call(system, prompt, max_new_tokens=300, temperature=0.1)

    if verbose: print(f"🤖 {answer}\n{'='*70}")
    return answer

# Demo
full_rag_answer("Thịt bò và thịt heo cái loại nào nhiều đạm hơn?")
full_rag_answer("Cơm trắng 100g có bao nhiêu calo?")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



👤 Thịt bò và thịt heo cái loại nào nhiều đạm hơn?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚙️  Intent:COMPARE | Lang:vi | top_k:8
🔀 Decomposed: ['Thịt bò có bao nhiêu gam đạm (protein) trên 100g?', 'Thịt heo có bao nhiêu gam đạm (protein) trên 100g?']


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🤖 Thịt bò (beef) có nhiều đạm hơn thịt heo (pork). Theo thông tin trong tài liệu, thịt bò có hàm lượng protein cao nhất là 23.1 g (từ Doc 5) và thấp nhất là 15.2 g (từ Doc 2), trung bình là 18.5 g. Còn thịt heo có hàm lượng protein cao nhất là 23.0 g (từ Doc 7) và thấp nhất là 17.3 g (từ Doc 8), trung bình là 20.3 g. Do đó, thịt bò có hàm lượng protein cao hơn thịt heo.

👤 Cơm trắng 100g có bao nhiêu calo?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚙️  Intent:LOOKUP | Lang:vi | top_k:3
🤖 Theo thông tin trong [CONTEXT], cơm trắng (gạo trắng) có 344.0 kcal/100g.


'Theo thông tin trong [CONTEXT], cơm trắng (gạo trắng) có 344.0 kcal/100g.'

In [ ]:
"""
=============================================================
PHẦN 3: SINH TEST SET  (fixed)
=============================================================
Bugs fixed:
  1. EN_TO_VI_MAP fallback về en_name → query tiếng Anh, router/retrieval fail
  2. parse_nutrients regex dùng raw-string escaped trong notebook → không match
  3. Subset tagging bị overwrite sau random.shuffle: retrieval_set lấy [:100]
     sau shuffle nên có thể toàn COMPARE/EXPLAIN, không có LOOKUP
  4. NUTRIENT_TEMPLATES query hỏi "phần trăm" đạm thay vì "gam" → numeric_em fail
"""

import os, json, random, re

TEST_SET_FILE = os.path.join(OUTPUT_DIR, "test_set.json")

# ── Fix 1: parse_nutrients dùng Python raw string đúng ──────
def parse_nutrients(text: str) -> dict:
    patterns = {
        "kcal":    r'(\d+(?:[.,]\d+)?)\s*kcal',
        "protein": r'(\d+(?:[.,]\d+)?)\s*g\b[^\n]*(?:protein|đạm)',
        "fat":     r'(\d+(?:[.,]\d+)?)\s*g\b[^\n]*(?:fat|béo|lipid)',
        "carbs":   r'(\d+(?:[.,]\d+)?)\s*g\b[^\n]*(?:carb|tinh bột)',
    }
    res = {}
    for k, pat in patterns.items():
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            res[k] = float(m.group(1).replace(",", "."))
    return res

def build_en_to_vi_map(vi_en_map):
    reverse_map = {}
    for vi, en in vi_en_map.items():
        en_lower = en.lower()
        # Giữ lại vi_name dài nhất (cụ thể nhất) thay vì ngắn nhất
        if en_lower not in reverse_map or len(vi) > len(reverse_map[en_lower]):
            reverse_map[en_lower] = vi
    return reverse_map

EN_TO_VI_MAP = build_en_to_vi_map(VI_EN_FOOD_MAP)

# ── Fix 2: Template không dùng "phần trăm", thêm nhiều mẫu tự nhiên hơn ──
NUTRIENT_TEMPLATES = {
    "kcal":    ["{food} 100g có bao nhiêu calo?",
                "Hàm lượng calo trong 100g {food} là bao nhiêu?",
                "100g {food} cung cấp bao nhiêu kcal?"],
    "protein": ["100g {food} có bao nhiêu gam đạm (protein)?",
                "{food} chứa bao nhiêu gam protein trong 100g?",
                "Hàm lượng protein của {food} là bao nhiêu?"],
    "fat":     ["Lượng chất béo trong 100g {food} là bao nhiêu gam?",
                "{food} có bao nhiêu gam fat trong 100g?"],
    "carbs":   ["{food} chứa bao nhiêu gam tinh bột (carbs) trong 100g?",
                "100g {food} có bao nhiêu gam carbs?"],
}

def generate_test_cases(records):
    cases = []
    lookup_candidates, compare_candidates, explain_candidates, recommend_candidates = [], [], [], []

    for r in records:
        rtype = r.get("record_type", "")
        if rtype == "food":
            lookup_candidates.append(r)
            compare_candidates.append(r)
        else:
            explain_candidates.append(r)
            recommend_candidates.append(r)

    # ── Fix 3: dùng food_name_vi trực tiếp nếu có, EN_TO_VI_MAP là fallback ──
    def get_vi_name(record):
        vi_direct = record.get("food_name_vi", "").strip()
        if vi_direct:
            return vi_direct
        en_name = str(record.get("food_name_en", "")).lower().strip()
        mapped = EN_TO_VI_MAP.get(en_name, "")
        if mapped:
            return mapped
        # cuối cùng mới dùng en_name (vẫn hỏi được, chỉ tiếng Anh)
        return en_name

    # 1. LOOKUP
    random.shuffle(lookup_candidates)
    for r in lookup_candidates:
        if len(cases) >= 250:
            break
        nutrients = parse_nutrients(r["text_for_embedding"])
        if not nutrients:
            continue
        vi_name = get_vi_name(r)
        nut_key = random.choice(list(nutrients.keys()))
        val = nutrients[nut_key]
        template = random.choice(NUTRIENT_TEMPLATES.get(nut_key, ["{food} chứa bao nhiêu " + nut_key + "?"]))
        question = template.format(food=vi_name)
        cases.append({
            "id": f"lookup_{len(cases):04d}",
            "intent": "LOOKUP",
            "query": question,
            "gold_answer": f"{val} {nut_key}",
            "gold_value": val,
            "nutrient": nut_key,
            "relevant_ids": [r["record_id"]],
        })

    # 2. COMPARE
    for _ in range(50):
        a, b = random.sample(lookup_candidates, 2)
        vi_a = get_vi_name(a)
        vi_b = get_vi_name(b)
        nut = random.choice(["protein", "kcal", "fat"])
        q = f"{vi_a.capitalize()} và {vi_b.lower()} loại nào nhiều {nut} hơn?"
        cases.append({
            "id": f"compare_{len(cases):04d}",
            "intent": "COMPARE",
            "query": q,
            "gold_answer": "So sánh " + nut,
            "gold_value": None,
            "nutrient": nut,
            "relevant_ids": [a["record_id"], b["record_id"]],
        })

    # 3. EXPLAIN
    for r in explain_candidates:
        if sum(1 for c in cases if c["intent"] == "EXPLAIN") >= 30:
            break
        name = r.get("group", "Chất này")
        q = f"Tại sao cơ thể cần {name} và vai trò của nó là gì?"
        cases.append({
            "id": f"explain_{len(cases):04d}",
            "intent": "EXPLAIN",
            "query": q,
            "gold_answer": r["text_for_embedding"][:150],
            "gold_value": None,
            "nutrient": name,
            "relevant_ids": [r["record_id"]],
        })

    # 4. RECOMMEND
    seen_rc = set()
    for _ in range(60):  # sample nhiều hơn để tránh lặp
        if sum(1 for c in cases if c["intent"] == "RECOMMEND") >= 30:
            break
        rc = random.choice(recommend_candidates)
        rid = rc["record_id"]
        if rid in seen_rc:
            continue
        seen_rc.add(rid)
        grp = rc.get("group", "Vi chất")
        age = rc.get("age", "người lớn")
        q = f"Làm sao để bổ sung đủ {grp} cho {age} qua ăn uống?"
        cases.append({
            "id": f"recommend_{len(cases):04d}",
            "intent": "RECOMMEND",
            "query": q,
            "gold_answer": "Khuyến nghị chế độ ăn",
            "gold_value": None,
            "nutrient": grp,
            "relevant_ids": [rc["record_id"]],
        })

    return cases

test_cases = generate_test_cases(all_records)

# ── Fix 4: Gán tag TRƯỚC khi shuffle để subset phân bổ đúng intent ──────────
# Router: 20 mỗi intent
router_set = []
for it in ["LOOKUP", "COMPARE", "EXPLAIN", "RECOMMEND"]:
    subs = [c for c in test_cases if c["intent"] == it][:20]
    for c in subs:
        c.setdefault("tags", [])
        if "router" not in c["tags"]:
            c["tags"].append("router")
    router_set.extend(subs)

# Numeric EM: 50 LOOKUP có gold_value
em_set = [c for c in test_cases if c["intent"] == "LOOKUP" and c.get("gold_value") is not None][:50]
for c in em_set:
    c.setdefault("tags", [])
    if "numeric_em" not in c["tags"]:
        c["tags"].append("numeric_em")

# Retrieval: 100 cases cân bằng intent
retrieval_set = []
per_intent = 25
for it in ["LOOKUP", "COMPARE", "EXPLAIN", "RECOMMEND"]:
    retrieval_set.extend([c for c in test_cases if c["intent"] == it][:per_intent])
for c in retrieval_set:
    c.setdefault("tags", [])
    if "retrieval" not in c["tags"]:
        c["tags"].append("retrieval")

# Text quality và RAGAS: lấy từ các case còn lại (không trong router/em)
tagged_ids = set(c["id"] for c in router_set + em_set + retrieval_set)

# ── Fix: Lọc riêng pool cho Text Quality, đảm bảo có gold_answer đủ dài ──
explain_lookup_pool = [
    c for c in test_cases
    if c["intent"] in ("EXPLAIN", "LOOKUP")
    and c["id"] not in tagged_ids
    and len(c.get("gold_answer", "")) > 20
]

tq_set = explain_lookup_pool[:30]
for c in tq_set:
    c.setdefault("tags", [])
    if "text_quality" not in c["tags"]:
        c["tags"].append("text_quality")

# Cập nhật lại tagged_ids sau khi lấy tq_set
tagged_ids.update(c["id"] for c in tq_set)

# Các cases còn lại dành cho RAGAS
remaining = [c for c in test_cases if c["id"] not in tagged_ids]
random.shuffle(remaining)

ragas_set = remaining[:50]
for c in ragas_set:
    c.setdefault("tags", [])
    if "ragas" not in c["tags"]:
        c["tags"].append("ragas")

# Khởi tạo tags cho tất cả case còn lại
for c in test_cases:
    c.setdefault("tags", [])

test_set = {
    "version": "v6_fixed", # Nên đổi version cho dễ track
    "total": len(test_cases),
    "cases": test_cases,
    "subsets": {
        "router": len(router_set),
        "retrieval": len(retrieval_set),
        "numeric_em": len(em_set),
        "text_quality": len(tq_set),
        "ragas": len(ragas_set),
    },
}

with open(TEST_SET_FILE, "w", encoding="utf-8") as f:
    json.dump(test_set, f, ensure_ascii=False, indent=2)

print(f"✅ test_set.json: {len(test_cases)} cases")
print(json.dumps(test_set["subsets"], indent=2))

# Sanity check: in phân bổ intent trong từng subset
for subset_name, subset in [("router", router_set), ("retrieval", retrieval_set), ("numeric_em", em_set)]:
    from collections import Counter
    dist = Counter(c["intent"] for c in subset)
    print(f"  [{subset_name}] intent dist: {dict(dist)}")

✅ test_set.json: 360 cases
{
  "router": 80,
  "retrieval": 100,
  "numeric_em": 50,
  "text_quality": 5,
  "ragas": 50
}
  [router] intent dist: {'LOOKUP': 20, 'COMPARE': 20, 'EXPLAIN': 20, 'RECOMMEND': 20}
  [retrieval] intent dist: {'LOOKUP': 25, 'COMPARE': 25, 'EXPLAIN': 25, 'RECOMMEND': 25}
  [numeric_em] intent dist: {'LOOKUP': 50}


In [ ]:
"""
=============================================================
PHẦN 4: RETRIEVAL VARIANTS (fixed)
=============================================================
Bugs fixed:
  1. `full` dùng use_hyde=False cho LOOKUP → embed query thô, bỏ lợi thế HyDE
     Fix: tách use_hyde ra khỏi variant logic, dùng mặc định True cho tất cả
  2. `full` rerank trên pool quá nhỏ (20 cands từ pool 30) → reranker loại nhầm
     Fix: tăng rrf_pool và max_rerank_candidates trong eval
  3. Không log progress → không biết variant nào bị treo
"""

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant) if relevant else 0.0

def mrr(retrieved, relevant):
    rel_set = set(relevant)
    for rank, did in enumerate(retrieved, 1):
        if did in rel_set:
            return 1.0 / rank
    return 0.0

def evaluate_retrieval_variants(test_cases, variants, top_k=10):
    subset = [c for c in test_cases if "retrieval" in c.get("tags", [])]
    results = {v: {"R@1": [], "R@3": [], "R@5": [], "MRR": []} for v in variants}

    print(f"Đang đánh giá {len(subset)} queries cho các variants: {variants}...")
    print(f"  Intent dist: {dict(Counter(c['intent'] for c in subset))}\n")

    for idx, tc in enumerate(subset):
        q = tc["query"]
        rel = tc["relevant_ids"]
        if not rel:
            continue

        for var in variants:
            # Fix: HyDE luôn bật với dense/rrf/full — không phân biệt intent
            # (intent routing là việc của pipeline, không phải của eval)
            use_hyde = var in ("dense", "rrf", "full")
            rerank   = (var == "full")

            top10 = retrieve_top_ids(
                q, variant=var, top_k=top_k,
                use_hyde=use_hyde, rerank=rerank
            )
            results[var]["R@1"].append(recall_at_k(top10, rel, 1))
            results[var]["R@3"].append(recall_at_k(top10, rel, 3))
            results[var]["R@5"].append(recall_at_k(top10, rel, 5))
            results[var]["MRR"].append(mrr(top10, rel))

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(subset)}] done")

    summary = {}
    print("\n| Variant | R@1  | R@3  | R@5  | MRR  |")
    print("|---------|------|------|------|------|")
    for v in variants:
        rs = results[v]
        r1 = sum(rs["R@1"]) / max(1, len(rs["R@1"]))
        r3 = sum(rs["R@3"]) / max(1, len(rs["R@3"]))
        r5 = sum(rs["R@5"]) / max(1, len(rs["R@5"]))
        m  = sum(rs["MRR"])  / max(1, len(rs["MRR"]))
        summary[v] = {"R@1": round(r1,3), "R@3": round(r3,3),
                      "R@5": round(r5,3), "MRR": round(m,3)}
        print(f"| {v:<7} | {r1:.2f} | {r3:.2f} | {r5:.2f} | {m:.2f} |")

    return summary

VARIANTS = ["bm25", "dense", "rrf", "full"]
retrieval_comparison = evaluate_retrieval_variants(test_set["cases"], VARIANTS)

with open(os.path.join(OUTPUT_DIR, "retrieval_comparison.json"), "w") as f:
    json.dump(retrieval_comparison, f, indent=2)

Đang đánh giá 100 queries cho các variants: ['bm25', 'dense', 'rrf', 'full']...
  Intent dist: {'LOOKUP': 25, 'COMPARE': 25, 'EXPLAIN': 25, 'RECOMMEND': 25}

  [10/100] done
  [20/100] done
  [30/100] done
  [40/100] done
  [50/100] done
  [60/100] done
  [70/100] done
  [80/100] done
  [90/100] done
  [100/100] done

| Variant | R@1  | R@3  | R@5  | MRR  |
|---------|------|------|------|------|
| bm25    | 0.55 | 0.85 | 0.94 | 0.79 |
| dense   | 0.47 | 0.78 | 0.88 | 0.70 |
| rrf     | 0.52 | 0.85 | 0.95 | 0.76 |
| full    | 0.47 | 0.74 | 0.79 | 0.67 |


In [33]:
"""
=============================================================
PHẦN 5.1: L1 ROUTER + L3 NUMERIC EM (fixed)
=============================================================
Bugs fixed:
  1. Router: LOOKUP support=0 vì sau shuffle, router_set lấy trước COMPARE.
     Fix: đã fix ở Phần 3 (tag trước shuffle)
  2. Numeric EM: query hỏi "phần trăm đạm" → LLM trả lời theo %, gold là gam
     → parse_nutrients không match → 0%. Fix: template đã sửa ở Phần 3
  3. Numeric EM: parse_nutrients chạy trên gold_answer ("23.5 protein") không có
     đơn vị "g" → regex không match. Fix: thêm fallback parse đơn giản hơn
"""

from sklearn.metrics import classification_report, f1_score
from collections import Counter

def evaluate_router(test_cases):
    subset = [c for c in test_cases if "router" in c.get("tags", [])]
    print(f"Router eval: {len(subset)} cases")
    print(f"  Intent dist: {dict(Counter(c['intent'] for c in subset))}")

    y_true = [c["intent"] for c in subset]
    y_pred = [route_query(c["query"]) for c in subset]

    print("\n=== Tầng 1: Router Evaluation ===")
    print(classification_report(
        y_true, y_pred,
        labels=["LOOKUP", "COMPARE", "RECOMMEND", "EXPLAIN"],
        zero_division=0
    ))
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    # Log các case bị predict sai để debug
    wrong = [(tc["query"], true, pred)
             for tc, true, pred in zip(subset, y_true, y_pred) if true != pred]
    if wrong:
        print(f"\n❌ Predict sai {len(wrong)} cases:")
        for q, t, p in wrong[:5]:
            print(f"  [{t}→{p}] {q[:70]}")

    return macro_f1

# Fix: parse gold_answer dạng "23.5 protein" (không có "g ")
def parse_gold_value(gold_answer: str, nutrient: str) -> float | None:
    """Parse gold_answer dạng '23.5 protein' hoặc '23.5 kcal'."""
    m = re.search(r'(\d+(?:[.,]\d+)?)', gold_answer)
    if m:
        return float(m.group(1).replace(",", "."))
    return None

def parse_pred_value(pred_text: str, nutrient: str) -> float | None:
    """Tìm số liệu của nutrient cụ thể trong câu trả lời."""
    # Ưu tiên pattern "X g protein/đạm" hoặc "X kcal"
    if nutrient == "kcal":
        patterns = [r'(\d+(?:[.,]\d+)?)\s*kcal', r'(\d+(?:[.,]\d+)?)\s*calo']
    else:
        nut_vi = {"protein": "đạm|protein", "fat": "béo|fat|lipid", "carbs": "tinh bột|carb"}
        kw = nut_vi.get(nutrient, nutrient)
        patterns = [
            rf'(\d+(?:[.,]\d+)?)\s*g\b[^\n]{{0,30}}(?:{kw})',
            rf'(?:{kw})[^\n]{{0,30}}?(\d+(?:[.,]\d+)?)\s*g\b',
        ]
    for pat in patterns:
        m = re.search(pat, pred_text, re.IGNORECASE)
        if m:
            return float(m.group(1).replace(",", "."))
    return None

def numeric_exact_match(pred: str, gold_answer: str, nutrient: str, tol: float = 0.05) -> bool:
    gval = parse_gold_value(gold_answer, nutrient)
    if gval is None:
        return False
    pval = parse_pred_value(pred, nutrient)
    if pval is None:
        return False
    if gval == 0:
        return pval == 0
    return abs(pval - gval) / gval <= tol

def evaluate_numeric_em(test_cases):
    subset = [c for c in test_cases if "numeric_em" in c.get("tags", [])]
    print(f"\n=== Tầng 3: Numeric EM ({len(subset)} queries) ===")

    total, correct = 0, 0
    failures = []

    for idx, tc in enumerate(subset):
        pred = full_rag_answer(tc["query"], verbose=False)
        nutrient = tc.get("nutrient", "kcal")

        # ── Fix: Bắt lỗi Ambiguity bằng cách đối chiếu record_id ──
        # Lấy top 3 context IDs từ pipeline để đối chiếu
        ctx_ids = retrieve_top_ids(tc["query"], variant="full", top_k=3)
        gold_id = tc["relevant_ids"][0] if tc.get("relevant_ids") else None

        # Rớt bài ngay nếu retrieval không vớt được đúng record chuẩn
        if gold_id and gold_id not in ctx_ids:
            ok = False
            failures.append((tc["query"], tc["gold_answer"], f"[Miss ID: {gold_id}] {pred[:50]}"))
        else:
            # Nếu retrieve đúng bài rồi thì mới check xem LLM extract số có đúng không
            ok = numeric_exact_match(pred, tc["gold_answer"], nutrient)
            if not ok:
                failures.append((tc["query"], tc["gold_answer"], pred[:80]))
        # ──────────────────────────────────────────────────────────

        total += 1
        if ok:
            correct += 1

        if idx < 3:
            print(f"Q: {tc['query']}")
            print(f"Gold: {tc['gold_answer']} | Match: {ok}")
            print(f"Pred (60c): {pred[:60]}...\n")

    score = correct / total if total else 0
    print(f"Numeric Exact Match (±5%): {score*100:.1f}%  ({correct}/{total})")

    if failures:
        print(f"\n❌ Fail cases ({min(3, len(failures))}/{len(failures)}):")
        for q, gold, pred in failures[:3]:
            print(f"  Q: {q[:60]} | Gold: {gold} | Pred: {pred}")

    return score

router_f1 = evaluate_router(test_set["cases"])
num_em    = evaluate_numeric_em(test_set["cases"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Router eval: 80 cases
  Intent dist: {'LOOKUP': 20, 'COMPARE': 20, 'EXPLAIN': 20, 'RECOMMEND': 20}


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Tầng 1: Router Evaluation ===
              precision    recall  f1-score   support

      LOOKUP       1.00      1.00      1.00        20
     COMPARE       1.00      1.00      1.00        20
   RECOMMEND       1.00      1.00      1.00        20
     EXPLAIN       1.00      1.00      1.00        20

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80


=== Tầng 3: Numeric EM (50 queries) ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Q: Hàm lượng calo trong 100g lê là bao nhiêu?
Gold: 45.0 kcal | Match: False
Pred (60c): Không có thông tin về hàm lượng calo trong 100g lê trong [CO...



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: papaya jam 100g có bao nhiêu calo?
Gold: 178.0 kcal | Match: True
Pred (60c): Theo thông tin trong [CONTEXT], papaya jam có 178.0 kcal/100...



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Numeric Exact Match (±5%): 92.0%  (46/50)

❌ Fail cases (3/4):
  Q: Hàm lượng calo trong 100g lê là bao nhiêu? | Gold: 45.0 kcal | Pred: Không có thông tin về hàm lượng calo trong 100g lê trong [CONTEXT]. (Note: There
  Q: Hàm lượng calo trong 100g soybean sauce with rice and maize  | Gold: 75.0 kcal | Pred: Theo thông tin trong [CONTEXT], hàm lượng calo trong 100g soybean sauce là 65.0 
  Q: ball shaped dumpling, vietnamese steamed buns 100g có bao nh | Gold: 219.0 kcal | Pred: Không có thông tin trong tài liệu về "ball shaped dumpling, vietnamese steamed b


In [34]:
"""
=============================================================
PHẦN 5.2: L4 TEXT QUALITY (fixed)
=============================================================
Bugs fixed:
  1. gold_answer của EXPLAIN chỉ lấy 100 ký tự đầu → quá ngắn, ROUGE/BLEU thấp
     Fix: đã tăng lên 150 ký tự ở Phần 3
  2. gold_answer của COMPARE/RECOMMEND là string cố định ("So sánh protein")
     → không có nghĩa để tính ROUGE. Fix: loại các case đó khỏi text_quality
  3. BLEU bị 0 vì ref quá ngắn + không dùng n-gram thích hợp
     Fix: dùng ROUGE-L + thêm F1 word overlap đơn giản hơn
"""

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_mod

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def word_f1(pred: str, ref: str) -> float:
    """F1 đơn giản trên token overlap (không cần stemmer)."""
    pred_tokens = set(pred.lower().split())
    ref_tokens  = set(ref.lower().split())
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = pred_tokens & ref_tokens
    p = len(common) / len(pred_tokens)
    r = len(common) / len(ref_tokens)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def evaluate_text_quality(test_cases):
    # Chỉ lấy case có gold_answer thực sự (EXPLAIN, một số LOOKUP)
    subset = [
        c for c in test_cases
        if "text_quality" in c.get("tags", [])
        and c["intent"] in ("EXPLAIN", "LOOKUP")
        and len(c.get("gold_answer", "")) > 20
    ]
    print(f"\n=== Tầng 4: Text Quality ({len(subset)} queries) ===")

    rscorer  = rs_mod.RougeScorer(["rougeL"], use_stemmer=False)
    smoother = SmoothingFunction().method1
    rouges, bleus, f1s = [], [], []

    for tc in subset:
        pred = full_rag_answer(tc["query"], verbose=False)
        ref  = tc["gold_answer"]

        rouge_l = rscorer.score(ref, pred)["rougeL"].fmeasure
        rt  = nltk.word_tokenize(ref.lower())
        pt  = nltk.word_tokenize(pred.lower())
        bl  = sentence_bleu([rt], pt, smoothing_function=smoother) if rt else 0.0
        wf1 = word_f1(pred, ref)

        rouges.append(rouge_l)
        bleus.append(bl)
        f1s.append(wf1)

    avg_rouge = sum(rouges) / max(1, len(rouges))
    avg_bleu  = sum(bleus)  / max(1, len(bleus))
    avg_f1    = sum(f1s)    / max(1, len(f1s))
    print(f"Avg ROUGE-L: {avg_rouge:.3f} | Avg BLEU: {avg_bleu:.3f} | Avg Word-F1: {avg_f1:.3f}")
    return avg_rouge, avg_bleu

text_rouge, text_bleu = evaluate_text_quality(test_set["cases"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Tầng 4: Text Quality (5 queries) ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Avg ROUGE-L: 0.042 | Avg BLEU: 0.008 | Avg Word-F1: 0.060


In [ ]:
"""
=============================================================
PHẦN 5.3: L5 RAGAS (fixed)
=============================================================
Bugs fixed:
  1. regex extract JSON an toàn hơn với try-except
  2. relevancy normalize /2 nhưng không clamp → có thể âm nếu model trả >2
  3. Không có retry khi Gemini rate limit → crash giữa chừng mất kết quả
  4. Không filter "ragas" cases có gold_answer vô nghĩa (COMPARE/RECOMMEND)
"""

import time
import json
import re
from google import genai
from google.colab import userdata
from tqdm import tqdm
import os

try:
    GENAI_API_KEY = userdata.get("GEMINI_API_KEY")
    client = genai.Client(api_key=GENAI_API_KEY)
except Exception:
    print("⚠️ Chưa cấu hình GEMINI_API_KEY trong Colab Secrets.")

def ragas_eval(query: str, answer: str, context: str, ground_truth: str = None) -> dict:
    prompt = (
        "Đánh giá câu trả lời dinh dưỡng. CHỈ trả về JSON hợp lệ, không có text khác:\n"
        '{"faithfulness": 0-1, "relevancy": 0-1, "context_precision": 0-1, "notes": "..."}\n'
        "- faithfulness: số liệu trong câu trả lời có khớp context không (0=bịa, 1=đúng)\n"
        "- relevancy: câu trả lời có đúng câu hỏi không (0=lạc đề, 1=đúng)\n"
        "- context_precision: tỷ lệ context được dùng hợp lý\n\n"
        f"[CÂU HỎI]: {query}\n"
        f"[CONTEXT]: {context[:1500]}\n"
        f"[CÂU TRẢ LỜI]: {answer[:800]}\n"
    )

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model='gemini-2.0-flash',
                contents=prompt,
            )
            text = response.text.strip()

            # Cải tiến regex để bắt trọn khối JSON dù có chứa ngoặc nhọn bên trong notes
            m = re.search(r'\{.*\}', text, re.DOTALL)
            if m:
                try:
                    parsed = json.loads(m.group(0))
                    for k in ("faithfulness", "relevancy", "context_precision"):
                        if k in parsed:
                            parsed[k] = max(0.0, min(1.0, float(parsed[k])))
                    return parsed
                except json.JSONDecodeError:
                    return {"error": "invalid_json", "raw": text[:200]}

            return {"error": "no_json", "raw": text[:200]}
        except Exception as e:
            if attempt < 2:
                time.sleep(5 * (attempt + 1))
            else:
                return {"error": str(e)}

def evaluate_layer5_ragas(test_cases):
    subset = [
        c for c in test_cases
        if "ragas" in c.get("tags", [])
        and c["intent"] in ("LOOKUP", "EXPLAIN")
    ]
    print(f"\n=== Tầng 5: RAGAS Eval ({len(subset)} queries) ===")

    ragas_file = os.path.join(OUTPUT_DIR, "ragas_results.jsonl")
    if os.path.exists(ragas_file):
        os.remove(ragas_file)

    results = []
    for tc in tqdm(subset):
        q   = tc["query"]
        ans = full_rag_answer(q, verbose=False)

        ctx_ids  = retrieve_top_ids(q, variant="full")
        ctx_text = ""
        if ctx_ids:
            res      = collection.get(ids=ctx_ids)
            ctx_text = "\n".join(res["documents"])[:VRAM_CONFIG["max_context_chars"]]

        metrics  = ragas_eval(q, ans, ctx_text, tc.get("gold_answer", ""))
        res_obj  = {"id": tc["id"], "query": q, "intent": tc["intent"], "metrics": metrics}
        results.append(res_obj)

        with open(ragas_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(res_obj, ensure_ascii=False) + "\n")

        time.sleep(1.5)  # Tránh Rate limit

    # Tổng hợp
    f_vals, r_vals, cp_vals = [], [], []
    errors = 0
    for r in results:
        m = r["metrics"]
        if "error" in m:
            errors += 1
            continue
        f_vals.append(float(m.get("faithfulness", 0)))
        r_vals.append(float(m.get("relevancy", 0)))
        cp_vals.append(float(m.get("context_precision", 0)))

    valid = len(f_vals)
    summary = {
        "faithfulness":      round(sum(f_vals) / valid  if valid else 0, 3),
        "relevancy":         round(sum(r_vals) / valid  if valid else 0, 3),
        "context_precision": round(sum(cp_vals) / valid if valid else 0, 3),
        "evaluated":         valid,
        "errors":            errors,
    }
    print("RAGAS Custom Summary:", summary)
    return summary

ragas_summary = evaluate_layer5_ragas(test_set["cases"])

In [36]:
"""
=============================================================
PHẦN 6: BÁO CÁO TỔNG HỢP (fixed)
=============================================================
"""

def generate_final_report():
    report = {
        "version": "v5_fixed",
        "targets": {
            "router_macro_f1 >= 0.85":        router_f1 >= 0.85,
            "numeric_em >= 0.75":              num_em >= 0.75,
            "text_rouge_l >= 0.40":            text_rouge >= 0.40,
            "ragas_faithfulness >= 0.80":      ragas_summary.get("faithfulness", 0) >= 0.80,
        },
        "scores": {
            "L1_router_macro_f1":       round(router_f1, 3),
            "L2_retrieval":             retrieval_comparison,
            "L3_numeric_em":            round(num_em, 3),
            "L4_rouge_l":               round(text_rouge, 3),
            "L4_bleu":                  round(text_bleu, 3),
            "L5_ragas":                 ragas_summary,
        },
    }

    report_file = os.path.join(OUTPUT_DIR, "eval_report_fixed.json")
    with open(report_file, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Báo cáo: {report_file}")
    print("\n=== Targets ===")
    for target, met in report["targets"].items():
        print(f"  {'✅' if met else '❌'} {target}")
    print("\n=== Scores ===")
    for k, v in report["scores"].items():
        if k != "L2_retrieval":
            print(f"  {k}: {v}")

generate_final_report()


✅ Báo cáo: /content/drive/MyDrive/nutrition_rag/outputs/v5/eval_report_fixed.json

=== Targets ===
  ✅ router_macro_f1 >= 0.85
  ✅ numeric_em >= 0.75
  ❌ text_rouge_l >= 0.40
  ❌ ragas_faithfulness >= 0.80

=== Scores ===
  L1_router_macro_f1: 1.0
  L3_numeric_em: 0.92
  L4_rouge_l: 0.042
  L4_bleu: 0.008
  L5_ragas: {'faithfulness': 0, 'relevancy': 0, 'context_precision': 0, 'evaluated': 0, 'errors': 45}
